# USDC Top Gainer predictor

In [1]:
%pip install lightgbm scikit-learn pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
# USDC Top Gainer Prediction (Notebook Style)

import requests
import pandas as pd
import time
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings
import joblib
warnings.filterwarnings("ignore")

# --------------------------
# 1. Collect USDC Pairs from Binbot API
# --------------------------
def get_usdc_pairs():
    r = requests.get("https://api.terminal.binbot.in/symbols")
    symbols = r.json()
    return [s["id"] for s in symbols["data"] if s["active"]]

# --------------------------
# 2. Fetch Kline Data
# --------------------------
def get_klines(symbol, interval="15m", limit=120):
    url = "https://api.binance.com/api/v3/klines"
    params = {"symbol": symbol, "interval": interval, "limit": limit}
    r = requests.get(url, params=params)
    data = r.json()
    df = pd.DataFrame(data, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"])
    df["close"] = df["close"].astype(float)
    df["volume"] = df["volume"].astype(float)
    df["open_time"] = pd.to_datetime(df["open_time"], unit='ms')
    return df

# --------------------------
# 3. Feature Engineering
# --------------------------
def compute_features(df):
    df["return_5m"] = df["close"].pct_change(5)
    df["return_15m"] = df["close"].pct_change(15)
    df["volatility"] = df["close"].rolling(10).std()
    df["ema5"] = df["close"].ewm(span=5).mean()
    df["ema15"] = df["close"].ewm(span=15).mean()
    df["ema_ratio"] = df["ema5"] / df["ema15"]
    df["volume_ratio"] = df["volume"] / df["volume"].rolling(15).mean()
    return df.iloc[-1][["return_5m", "return_15m", "volatility", "ema_ratio", "volume_ratio"]]

# --------------------------
# 4. Build Live Feature Set
# --------------------------
def build_live_features():
    usdc_pairs = get_usdc_pairs()
    rows = []
    for symbol in usdc_pairs:
        try:
            df = get_klines(symbol)
            features = compute_features(df)
            features["symbol"] = symbol
            features["current_price"] = df["close"].iloc[-1]
            rows.append(features)
            time.sleep(0.1)
        except Exception as e:
            print(f"{symbol} failed: {e}")
    return pd.DataFrame(rows)

# --------------------------
# 5. Train Model on Historical Data
# --------------------------
def generate_training_data(pairs, history=180):
    data = []
    for symbol in pairs:
        try:
            df = get_klines(symbol, limit=history)
            for i in range(60, len(df)-60):
                window = df.iloc[i-60:i]
                features = compute_features(window)
                price_now = df.iloc[i]["close"]
                price_future = df.iloc[i+60]["close"]
                target = 1 if (price_future - price_now) / price_now > 0.05 else 0  # 5% gain
                features["symbol"] = symbol
                features["target"] = target
                data.append(features)
            time.sleep(0.2)
        except Exception as e:
            print(f"[Training] {symbol} failed: {e}")
    return pd.DataFrame(data)

# --------------------------
# 6. Train Model
# --------------------------
def train_model(df):
    X = df.drop(["symbol", "target"], axis=1)
    y = df["target"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
    model = LGBMClassifier()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    return model

# --------------------------
# 7. Run Live Predictions
# --------------------------
def predict_top_gainers(model):
    df = build_live_features()
    features = df.drop(["symbol", "current_price"], axis=1)
    probs = model.predict_proba(features)[:, 1]
    df["score"] = probs
    top10 = df.sort_values("score", ascending=False).head(10)
    return top10[["symbol", "score", "current_price"]]

# --------------------------
# RUN PIPELINE
# --------------------------
print("1. Collecting pairs and training data...")
usdc_pairs = get_usdc_pairs()
train_df = generate_training_data(usdc_pairs[:10])  # limit to 10 pairs for speed

print("2. Training model...")
model = train_model(train_df)

print("3. Running live predictions...")
top_preds = predict_top_gainers(model)
print(top_preds)

print("4. Save model...")
joblib.dump(model, "./checkpoints/usdc_gainer_model.pkl")


1. Collecting pairs and training data...
2. Training model...
[LightGBM] [Info] Number of positive: 70, number of negative: 410
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000286 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 728
[LightGBM] [Info] Number of data points in the train set: 480, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.145833 -> initscore=-1.767662
[LightGBM] [Info] Start training from score -1.767662
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Li

['./checkpoints/usdc_gainer_model.pkl']

# Production model

In [11]:
# USDC Top Gainer Prediction (Production Version)

import requests
import pandas as pd
import time
import joblib
import warnings
warnings.filterwarnings("ignore")

MODEL_PATH = "./checkpoints/usdc_gainer_model.pkl"

# 1. Collect USDC Pairs from Binbot API
def get_usdc_pairs():
    r = requests.get("https://api.terminal.binbot.in/symbols")
    symbols = r.json()
    return [s["id"] for s in symbols["data"] if s["quote_asset"] == "USDC" and s["active"] == True]

# 2. Fetch Kline Data
def get_klines(symbol, interval="1m", limit=120):
    url = "https://api.binance.com/api/v3/klines"
    params = {"symbol": symbol, "interval": interval, "limit": limit}
    r = requests.get(url, params=params)
    data = r.json()
    df = pd.DataFrame(data, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"])
    df["close"] = df["close"].astype(float)
    df["volume"] = df["volume"].astype(float)
    df["open_time"] = pd.to_datetime(df["open_time"], unit='ms')
    return df

# 3. Feature Engineering
def compute_features(df):
    df["return_5m"] = df["close"].pct_change(5)
    df["return_15m"] = df["close"].pct_change(15)
    df["volatility"] = df["close"].rolling(10).std()
    df["ema5"] = df["close"].ewm(span=5).mean()
    df["ema15"] = df["close"].ewm(span=15).mean()
    df["ema_ratio"] = df["ema5"] / df["ema15"]
    df["volume_ratio"] = df["volume"] / df["volume"].rolling(15).mean()
    return df.iloc[-1][["return_5m", "return_15m", "volatility", "ema_ratio", "volume_ratio"]]

# 4. Build Live Feature Set
def build_live_features():
    usdc_pairs = get_usdc_pairs()
    rows = []
    for symbol in usdc_pairs:
        try:
            df = get_klines(symbol)
            features = compute_features(df)
            features["symbol"] = symbol
            features["current_price"] = df["close"].iloc[-1]
            rows.append(features)
            time.sleep(0.1)
        except Exception:
            continue
    return pd.DataFrame(rows)


# 6. Run Live Predictions
def predict_top_gainers(model):
    df = build_live_features()
    features = df.drop(["symbol", "current_price"], axis=1)
    probs = model.predict_proba(features)[:, 1]
    df["score"] = probs
    top10 = df.sort_values("score", ascending=False).head(10)
    return top10[["symbol", "score", "current_price"]]

# MAIN
model = joblib.load(MODEL_PATH)
top_preds = predict_top_gainers(model)
print(top_preds)


        symbol     score  current_price
119   AAVEUSDC  0.992923     319.870000
119    MKRUSDC  0.852821    1990.000000
119    TAOUSDC  0.822911     408.800000
119    TRXUSDC  0.649221       0.325100
119    SOLUSDC  0.607670     176.200000
119  TURBOUSDC  0.558655       0.005370
119   DOGEUSDC  0.555175       0.239500
119    FTMUSDC  0.472387       0.698600
119  PENGUUSDC  0.467519       0.031092
119    SXTUSDC  0.202409       0.085400
